# AI Trading Bot — Colab Training

Run this notebook on Google Colab (T4 or A100 GPU).
**Do NOT run locally** — torch/stable-baselines3 are Colab-only dependencies.

Before running: add `GEMINI_API_KEY` and `NEWS_API_KEY` to Colab Secrets (key icon in left sidebar).

In [ ]:
# Cell 1 — Setup: clone repo and install training dependencies
!git clone https://github.com/YashKasare21/trading_bot.git
%cd trading_bot
!pip install uv -q
!uv pip install -e '.[training]' -q

In [ ]:
# Cell 2 — Mount Google Drive and load secrets
from google.colab import drive, userdata
import os

drive.mount('/content/drive')

os.environ['GEMINI_API_KEY'] = userdata.get('GEMINI_API_KEY')
os.environ['NEWS_API_KEY'] = userdata.get('NEWS_API_KEY')

MODEL_SAVE_DIR = '/content/drive/MyDrive/trading_bot/models/'
os.makedirs(MODEL_SAVE_DIR, exist_ok=True)
print(f'Model save directory: {MODEL_SAVE_DIR}')

In [ ]:
# Cell 3 — Fetch data and build features
from trading_bot.data.fetcher import MarketDataFetcher
from trading_bot.data.sentiment import SentimentAnalyzer
from trading_bot.features.pipeline import FeaturePipeline
from pathlib import Path
from datetime import date

fetcher = MarketDataFetcher()
df = fetcher.fetch_ohlcv('^NSEI', start=date(2018, 1, 1), end=date(2024, 12, 31))
print(f'Raw OHLCV shape: {df.shape}')

analyzer = SentimentAnalyzer()
sentiment_df = analyzer.fetch_and_score_news('^NSEI', start=date(2018, 1, 1), end=date(2024, 12, 31))
print(f'Sentiment shape: {sentiment_df.shape}')

pipeline = FeaturePipeline(window_size=20, fit_regime=True)
featured_df = pipeline.fit_transform(df, sentiment_df)
pipeline.save(Path(MODEL_SAVE_DIR) / 'pipeline_config.joblib')

print(f'Training data shape: {featured_df.shape}')
print(f'Feature count: {len(pipeline.get_feature_names())}')
print(f'Features: {pipeline.get_feature_names()[:10]} ...')

In [ ]:
# Cell 4 — Train PPO agent
from stable_baselines3 import PPO
from stable_baselines3.common.vec_env import DummyVecEnv, VecNormalize
from stable_baselines3.common.callbacks import CheckpointCallback
from trading_bot.env.trading_env import StockTradingEnv

train_df = featured_df.iloc[:int(len(featured_df) * 0.8)].copy()

env = DummyVecEnv([lambda: StockTradingEnv(train_df, window_size=20)])
env = VecNormalize(env, norm_obs=True, norm_reward=True)

checkpoint = CheckpointCallback(
    save_freq=50_000,
    save_path=MODEL_SAVE_DIR,
    name_prefix='ppo_nifty',
)

model = PPO(
    'MlpPolicy', env, verbose=1,
    learning_rate=3e-4, n_steps=2048, batch_size=64,
    tensorboard_log='/content/logs/',
)
model.learn(total_timesteps=500_000, callback=checkpoint)
model.save(f'{MODEL_SAVE_DIR}/ppo_nifty_final')
env.save(f'{MODEL_SAVE_DIR}/vec_normalize_ppo.pkl')
print('PPO training complete.')

In [ ]:
# Cell 5 — Train SAC agent (primary algo per CLAUDE.md)
from stable_baselines3 import SAC

sac_env = DummyVecEnv([lambda: StockTradingEnv(train_df, window_size=20)])
sac_env = VecNormalize(sac_env, norm_obs=True, norm_reward=True)

sac_model = SAC(
    'MlpPolicy', sac_env, verbose=1,
    learning_rate=3e-4, buffer_size=100_000,
    tensorboard_log='/content/logs/',
)
sac_model.learn(total_timesteps=500_000)
sac_model.save(f'{MODEL_SAVE_DIR}/sac_nifty_final')
sac_env.save(f'{MODEL_SAVE_DIR}/vec_normalize_sac.pkl')
print('SAC training complete.')